###Inicialización del Entorno y Estructura SCD2

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- Crear la Tabla de Dimensión Clientes con lógica SCD Tipo 2
-- cliente_sk: Llave subrogada 
CREATE TABLE IF NOT EXISTS workspace.gold.dim_clientes (
    cliente_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    cliente_id INT,
    nombre_cliente STRING,
    ciudad STRING,
    valido_desde DATE,
    valido_hasta DATE,
    es_actual BOOLEAN
) USING DELTA;

ALTER TABLE workspace.gold.dim_clientes SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

###Lógica de Carga Incremental SCD2

In [0]:
%sql
-- 3. Lógica de Inserción para gestionar la Dimensión Histórica
INSERT INTO workspace.gold.dim_clientes (cliente_id, nombre_cliente, ciudad, valido_desde, valido_hasta, es_actual)
SELECT 
    CAST(cliente_id AS INT),
    nombre AS nombre_cliente,
    ciudad,
    CURRENT_DATE() AS valido_desde,
    CAST('9999-12-31' AS DATE) AS valido_hasta,
    TRUE AS es_actual
FROM workspace.silver.clientes -- Única fuente de verdad autorizada
WHERE CAST(cliente_id AS INT) NOT IN (
    SELECT cliente_id 
    FROM workspace.gold.dim_clientes 
    WHERE es_actual = TRUE
);


###Construcción y Carga de la Tabla de Hechos

In [0]:
%sql
-- 4. Crear la Tabla de Hechos de Ventas
CREATE TABLE IF NOT EXISTS workspace.gold.fact_ventas (
    venta_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    cliente_sk BIGINT, 
    producto STRING,
    fecha_venta DATE,
    pais STRING,
    cantidad INT,
    valor_unitario DOUBLE,
    valor_total DOUBLE
) USING DELTA;

-- 5. Carga incremental de Hechos cruzando Silver Ventas con la Dimensión Gold activa
INSERT INTO workspace.gold.fact_ventas (cliente_sk, producto, fecha_venta, pais, cantidad, valor_unitario, valor_total)
SELECT 
    d.cliente_sk,
    s.producto,
    CAST(s.fecha_venta AS DATE),
    s.pais,
    s.cantidad,
    s.valor_unitario,
    s.valor_total
FROM workspace.silver.ventas s
-- JOIN fundamental: Conectamos el ID transaccional (Silver) con la llave subrogada (Gold)
JOIN workspace.gold.dim_clientes d 
    ON s.cliente_id = d.cliente_id 
    AND d.es_actual = TRUE
-- Filtro para evitar re-procesar ventas
WHERE CAST(s.fecha_venta AS DATE) > (
    SELECT COALESCE(MAX(fecha_venta), '1900-01-01') 
    FROM workspace.gold.fact_ventas
);
